<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [2]:
!pip install pandas
!pip install matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [3]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [4]:
# Create a  dataset summary
summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().mul(100).round(2).values
    ),
    "Unique_Values": df.nunique(dropna=True).values
})

summary = summary.sort_values(
    by="Missing_Percentage",
    ascending=False
).reset_index(drop=True)

display(summary)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage,Unique_Values
0,AINextMuch less integrated,str,1148,64289,98.25,286
1,AINextLess integrated,str,2355,63082,96.40,249
2,AINextNo change,str,12498,52939,80.90,539
3,AINextMuch more integrated,str,13438,51999,79.46,700
4,EmbeddedAdmired,str,16733,48704,74.43,1053
...,...,...,...,...,...,...
109,MainBranch,str,65437,0,0.00,5
110,Age,str,65437,0,0.00,8
111,Employment,str,65437,0,0.00,110
112,Check,str,65437,0,0.00,1


In [5]:
summary["Column_Type"] = summary["Data_Type"].apply(
    lambda x: "Numeric"
    if "int" in x or "float" in x
    else "Categorical/Text"
)

display(summary)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage,Unique_Values,Column_Type
0,AINextMuch less integrated,str,1148,64289,98.25,286,Categorical/Text
1,AINextLess integrated,str,2355,63082,96.40,249,Categorical/Text
2,AINextNo change,str,12498,52939,80.90,539,Categorical/Text
3,AINextMuch more integrated,str,13438,51999,79.46,700,Categorical/Text
4,EmbeddedAdmired,str,16733,48704,74.43,1053,Categorical/Text
...,...,...,...,...,...,...,...
109,MainBranch,str,65437,0,0.00,5,Categorical/Text
110,Age,str,65437,0,0.00,8,Categorical/Text
111,Employment,str,65437,0,0.00,110,Categorical/Text
112,Check,str,65437,0,0.00,1,Categorical/Text


In [6]:
summary.to_csv(
    "dataset_quality_summary.csv",
    index=False
)

In [7]:
missing_summary = summary[
    summary["Missing_Count"] > 0
].reset_index(drop=True)

display(missing_summary)

,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Percentage,Unique_Values,Column_Type
0,AINextMuch less integrated,str,1148,64289,98.25,286,Categorical/Text
1,AINextLess integrated,str,2355,63082,96.40,249,Categorical/Text
2,AINextNo change,str,12498,52939,80.90,539,Categorical/Text
3,AINextMuch more integrated,str,13438,51999,79.46,700,Categorical/Text
4,EmbeddedAdmired,str,16733,48704,74.43,1053,Categorical/Text
...,...,...,...,...,...,...,...
104,YearsCode,str,59869,5568,8.51,52,Categorical/Text
105,NEWSOSites,str,60286,5151,7.87,32,Categorical/Text
106,LearnCode,str,60488,4949,7.56,418,Categorical/Text
107,EdLevel,str,60784,4653,7.11,8,Categorical/Text


<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [8]:
numeric_summary = (
    df.select_dtypes(include="number")
      .describe()
      .T
      .rename(columns={
          "count": "Count",
          "mean": "Mean",
          "std": "Std_Dev",
          "min": "Minimum",
          "25%": "Q1",
          "50%": "Median",
          "75%": "Q3",
          "max": "Maximum"
      })
      .round(2)
)

display(numeric_summary)

,Count,Mean,Std_Dev,Minimum,Q1,Median,Q3,Maximum
ResponseId,65437.0,3.271900e+04,1.889018e+04,1.0,16360.0,32719.0,49078.0,6.543700e+04
CompTotal,33740.0,2.963841e+145,5.444117e+147,0.0,60000.0,110000.0,250000.0,1.000000e+150
WorkExp,29658.0,1.147000e+01,9.170000e+00,0.0,4.0,9.0,16.0,5.000000e+01
JobSatPoints_1,29324.0,1.858000e+01,2.597000e+01,0.0,0.0,10.0,22.0,1.000000e+02
JobSatPoints_4,29393.0,7.520000e+00,1.842000e+01,0.0,0.0,0.0,5.0,1.000000e+02
JobSatPoints_5,29411.0,1.006000e+01,2.183000e+01,0.0,0.0,0.0,10.0,1.000000e+02
JobSatPoints_6,29450.0,2.434000e+01,2.709000e+01,0.0,0.0,20.0,30.0,1.000000e+02
JobSatPoints_7,29448.0,2.297000e+01,2.702000e+01,0.0,0.0,15.0,30.0,1.000000e+02
JobSatPoints_8,29456.0,2.028000e+01,2.611000e+01,0.0,0.0,10.0,25.0,1.000000e+02
JobSatPoints_9,29456.0,1.617000e+01,2.485000e+01,0.0,0.0,5.0,20.0,1.000000e+02


### Analysis

The descriptive statistics reveal that the dataset is generally well structured, but several preprocessing steps are required before analysis. Compensation-related variables contain a high proportion of missing values and extreme outliers, which distort summary statistics such as the mean. In contrast, variables like `WorkExp` and `JobSat` appear consistent and within expected ranges. These results indicate that the next stages of data wrangling should focus on handling missing values, validating compensation records, treating outliers, and normalizing salary-related features to improve data quality and ensure more reliable analyses.

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [9]:
country_audit = (
    df["Country"]
      .value_counts(dropna=False)
      .rename_axis("Country")
      .reset_index(name="Count")
)

country_audit["Percentage"] = (
    country_audit["Count"] / len(df) * 100
).round(2)

display(country_audit)

,Country,Count,Percentage
0,United States of America,11095,16.96
1,NaN,6507,9.94
2,Germany,4947,7.56
3,India,4231,6.47
4,United Kingdom of Great Britain and Northern I...,3224,4.93
...,...,...,...
181,Haiti,1,0.00
182,Nauru,1,0.00
183,Chad,1,0.00
184,Djibouti,1,0.00


In [10]:
# Profile the Country column
country_profile = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Non-Null Values",
        "Missing Values",
        "Missing Percentage (%)",
        "Unique Countries"
    ],
    "Value": [
        len(df),
        df["Country"].notna().sum(),
        df["Country"].isna().sum(),
        round(df["Country"].isna().mean() * 100, 2),
        df["Country"].nunique()
    ]
})

display(country_profile)

,Metric,Value
0,Total Records,65437.00
1,Non-Null Values,58930.00
2,Missing Values,6507.00
3,Missing Percentage (%),9.94
4,Unique Countries,185.00


In [11]:
# Country frequency distribution
country_distribution = (
    df["Country"]
    .value_counts(dropna=False)
    .rename_axis("Country")
    .reset_index(name="Count")
)

country_distribution["Percentage (%)"] = (
    country_distribution["Count"] / len(df) * 100
).round(2)

display(country_distribution)

,Country,Count,Percentage (%)
0,United States of America,11095,16.96
1,NaN,6507,9.94
2,Germany,4947,7.56
3,India,4231,6.47
4,United Kingdom of Great Britain and Northern I...,3224,4.93
...,...,...,...
181,Haiti,1,0.00
182,Nauru,1,0.00
183,Chad,1,0.00
184,Djibouti,1,0.00


In [12]:
# Detect leading/trailing spaces
country_spaces = df[
    df["Country"].notna() &
    (df["Country"] != df["Country"].str.strip())
][["Country"]].drop_duplicates()

display(country_spaces)

,Country


In [13]:
# Detect possible case inconsistencies (Uppercase diferentiation)
country_case = (
    pd.DataFrame({
        "Original": df["Country"].dropna(),
        "Normalized": df["Country"].dropna().str.lower()
    })
)

display(country_case.groupby("Normalized")["Original"].unique())

Normalized
afghanistan                                                      [Afghanistan]
albania                                                              [Albania]
algeria                                                              [Algeria]
andorra                                                              [Andorra]
angola                                                                [Angola]
                                                         ...                  
venezuela, bolivarian republic of...    [Venezuela, Bolivarian Republic of...]
viet nam                                                            [Viet Nam]
yemen                                                                  [Yemen]
zambia                                                                [Zambia]
zimbabwe                                                            [Zimbabwe]
Name: Original, Length: 185, dtype: object

In [14]:
# Detect missing or blank values
irrelevant_entries = df[
    df["Country"].isna() |
    (df["Country"].astype(str).str.strip() == "")
]

display(irrelevant_entries[["Country"]])

,Country
43448,NaN
43454,NaN
43459,NaN
43460,NaN
43461,NaN
...,...
65430,NaN
65432,NaN
65433,NaN
65434,NaN


In [15]:
blank_strings = (
    df["Country"].notna() &
    (df["Country"].str.strip() == "")
).sum()

In [16]:
audit_summary = pd.DataFrame({
    "Issue": [
        "Missing Values",
        "Leading/Trailing Spaces",
        "Blank Strings"
    ],
    "Count": [
        df["Country"].isna().sum(),
        len(country_spaces),
        blank_strings
    ]
})

display(audit_summary)

,Issue,Count
0,Missing Values,6507
1,Leading/Trailing Spaces,0
2,Blank Strings,0


### Analysis

The `Country` column was audited to identify potential data quality issues before applying any transformations. The audit revealed **6,507 missing values**, representing records where no country information was provided. However, no leading or trailing spaces and no blank string entries were detected, indicating that the existing country names are consistently formatted.

These findings suggest that the primary issue affecting this column is missing data rather than formatting inconsistencies. Therefore, no standardization or text-cleaning operations are required for the valid country entries. The focus of subsequent preprocessing should be on handling the missing values using an appropriate imputation or exclusion strategy, depending on the analysis objectives.

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [17]:
# Install the libraries only if they are not already available.
# Uncomment the following line when running in a new environment.
!pip install rapidfuzz pycountry

import re
import unicodedata

import pandas as pd
import pycountry

from rapidfuzz import fuzz, process


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
def normalize_country_text(value):
    """
    Normalize a country name for comparison purposes.

    The function:
    - Preserves missing values as None.
    - Converts text to lowercase.
    - Applies Unicode normalization.
    - Removes accents and diacritical marks.
    - Replaces punctuation with spaces.
    - Collapses repeated whitespace.

    Parameters
    ----------
    value : object
        Original country value.

    Returns
    -------
    str or None
        Normalized country text used only for matching.
    """
    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    # Decompose accented characters, e.g. "é" -> "e" + accent mark.
    text = unicodedata.normalize("NFKD", text)

    # Remove the decomposed accent marks.
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    # Replace punctuation and special characters with spaces.
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Remove repeated and leading/trailing whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [19]:
#Create a frequency table before modifying the original column.
country_profile = (
    df["Country"]
    .value_counts(dropna=False)
    .rename_axis("Original_Country")
    .reset_index(name="Count")
)

country_profile["Percentage"] = (
    country_profile["Count"]
    .div(len(df))
    .mul(100)
    .round(2)
)

display(country_profile)

,Original_Country,Count,Percentage
0,United States of America,11095,16.96
1,NaN,6507,9.94
2,Germany,4947,7.56
3,India,4231,6.47
4,United Kingdom of Great Britain and Northern I...,3224,4.93
...,...,...,...
181,Haiti,1,0.00
182,Nauru,1,0.00
183,Chad,1,0.00
184,Djibouti,1,0.00


In [20]:
# Create a clean list of available non-null country names.
available_countries = (
    country_profile["Original_Country"]
    .dropna()
    .sort_values()
    .reset_index(drop=True)
)

with pd.option_context("display.max_rows", None):
    display(available_countries.to_frame(name="Country"))

,Country
0,Afghanistan
1,Albania
2,Algeria
3,Andorra
4,Angola
5,Antigua and Barbuda
6,Argentina
7,Armenia
8,Australia
9,Austria


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [21]:
# Inspect the distinct Employment categories before encoding.
# This helps validate how many dummy variables will be created.

employment_profile = (
    df["Employment"]
    .value_counts(dropna=False)
    .rename_axis("Employment")
    .reset_index(name="Count")
)

display(employment_profile)

,Employment,Count
0,"Employed, full-time",39041
1,"Independent contractor, freelancer, or self-em...",4846
2,"Student, full-time",4709
3,"Employed, full-time;Independent contractor, fr...",3557
4,"Not employed, but looking for work",2341
...,...,...
105,"Not employed, but looking for work;Independent...",1
106,"Student, full-time;Retired",1
107,"Employed, full-time;Not employed, but looking ...",1
108,"Not employed, and not looking for work;Student...",1


In [28]:
# Apply one-hot encoding to the Employment column.
#
# Each distinct employment category becomes a binary column:
#   1 -> the respondent belongs to that category
#   0 -> the respondent does not belong to that category
#
# dtype=int is used so the encoded values are stored as 0/1
# instead of Boolean True/False.

employment_encoded = pd.get_dummies(
    df["Employment"],
    prefix="Employment",
    dtype=int
)

# Preserve the original Employment column for interpretability
# and append the encoded features to the dataset.

df = pd.concat(
    [df, employment_encoded],
    axis=1
)

display(employment_encoded.head())

print(
    f"Employment categories encoded: "
    f"{employment_encoded.shape[1]}"
)

,"Employment_Employed, full-time","Employment_Employed, full-time;Employed, part-time","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time;Retired","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Employed, part-time","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Not employed, and not looking for work;Student, part-time","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Retired","Employment_Employed, full-time;Independent contractor, freelancer, or self-employed;Student, part-time",...,"Employment_Student, full-time;Not employed, but looking for work;Not employed, and not looking for work;Student, part-time","Employment_Student, full-time;Not employed, but looking for work;Retired","Employment_Student, full-time;Not employed, but looking for work;Student, part-time","Employment_Student, full-time;Retired","Employment_Student, full-time;Student, part-time","Employment_Student, full-time;Student, part-time;Employed, part-time","Employment_Student, full-time;Student, part-time;Retired","Employment_Student, part-time","Employment_Student, part-time;Employed, part-time","Employment_Student, part-time;Retired"
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Employment categories encoded: 110


### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [29]:
# Build a missing-value quality report for all dataset columns.
#
# The report includes:
# - Number of missing observations
# - Percentage of missing observations
#
# Columns are sorted from highest to lowest missingness
# to prioritize data-cleaning decisions.

missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean()
        .mul(100)
        .round(2)
        .values
    )
})

missing_summary = (
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ]
    .sort_values(
        "Missing_Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_summary)

,Column,Missing_Count,Missing_Percentage
0,AINextMuch less integrated,64289,98.25
1,AINextLess integrated,63082,96.40
2,AINextNo change,52939,80.90
3,AINextMuch more integrated,51999,79.46
4,EmbeddedAdmired,48704,74.43
...,...,...,...
104,YearsCode,5568,8.51
105,NEWSOSites,5151,7.87
106,LearnCode,4949,7.56
107,EdLevel,4653,7.11


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [30]:
# Inspect the amount of missing compensation data before imputation.

comp_missing_before = (
    df["ConvertedCompYearly"]
    .isna()
    .sum()
)

comp_median = (
    df["ConvertedCompYearly"]
    .median()
)

print(
    "Missing values before imputation:",
    comp_missing_before
)

print(
    "Median annual compensation:",
    comp_median
)

Missing values before imputation: 42002
Median annual compensation: 65000.0


In [32]:
# Impute missing annual compensation values using the median.
#
# The median is preferred over the mean because the compensation
# distribution is strongly right-skewed and contains extreme outliers.
# Unlike the mean, the median is considerably less affected by
# exceptionally high salary observations.

df["ConvertedCompYearly"] = (
    df["ConvertedCompYearly"]
    .fillna(comp_median)
)

# Validate that the imputation was successfully applied.

comp_missing_after = (
    df["ConvertedCompYearly"]
    .isna()
    .sum()
)

print(
    "Missing values after imputation:",
    comp_missing_after
)

Missing values after imputation: 0


In [33]:
# Verify that only missing values were treated
# and that the column remains numeric.

validation = pd.DataFrame({
    "Metric": [
        "Missing Before",
        "Missing After",
        "Imputed Values",
        "Median Used"
    ],
    "Value": [
        comp_missing_before,
        comp_missing_after,
        comp_missing_before - comp_missing_after,
        comp_median
    ]
})

display(validation)

,Metric,Value
0,Missing Before,42002.0
1,Missing After,0.0
2,Imputed Values,42002.0
3,Median Used,65000.0


### Compensation Imputation Analysis

`ConvertedCompYearly` contains a substantial amount of missing data. Because the variable is highly right-skewed and includes extreme salary outliers, median imputation was selected instead of the mean. The median provides a more robust estimate of a typical annual compensation value and is less influenced by exceptionally high observations. After imputation, the column was validated to confirm that no missing compensation values remained.

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [34]:
# Profile RemoteWork before applying categorical imputation.
remote_profile = (
    df["RemoteWork"]
    .value_counts(dropna=False)
    .rename_axis("RemoteWork")
    .reset_index(name="Count")
)

remote_profile["Percentage"] = (
    remote_profile["Count"] / len(df) * 100
).round(2)

display(remote_profile)

,RemoteWork,Count,Percentage
0,"Hybrid (some remote, some in-person)",23015,35.17
1,Remote,20831,31.83
2,In-person,10960,16.75
3,NaN,10631,16.25


In [35]:
# Identify the most frequent non-null category.
remote_mode = df["RemoteWork"].mode()[0]

# Store the number of missing values before imputation.
missing_before = df["RemoteWork"].isna().sum()

print("Most frequent category:", remote_mode)
print("Missing values before imputation:", missing_before)

# Replace missing values with the most frequent category.
df["RemoteWork"] = df["RemoteWork"].fillna(remote_mode)

# Validate the result.
missing_after = df["RemoteWork"].isna().sum()

print("Missing values after imputation:", missing_after)
print("Values imputed:", missing_before - missing_after)

Most frequent category: Hybrid (some remote, some in-person)
Missing values before imputation: 10631
Missing values after imputation: 0
Values imputed: 10631


### RemoteWork Imputation Analysis

Missing values in `RemoteWork` were imputed using the most frequent non-null category (mode). This approach preserves the categorical structure of the variable without introducing a new artificial category. The column was validated after imputation to confirm that no missing values remained.

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [36]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the Min-Max scaler.
# Min-Max scaling transforms the values into the [0, 1] range.
scaler = MinMaxScaler()

# Scale annual compensation while preserving the original variable.
df["ConvertedCompYearly_MinMax"] = scaler.fit_transform(
    df[["ConvertedCompYearly"]]
)

# Validate the resulting range.
print(
    df["ConvertedCompYearly_MinMax"]
    .agg(["min", "max"])
)

min    0.0
max    1.0
Name: ConvertedCompYearly_MinMax, dtype: float64


### Analysis

The `RemoteWork` column was reviewed before imputation to identify its most frequent valid category. Missing values were then replaced using the mode, which is an appropriate strategy for categorical variables when preserving the existing category distribution is preferred. After the transformation, the column was validated to confirm that no missing values remained. This approach avoids introducing arbitrary numerical values or unrelated categories into a categorical feature.

<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [37]:
import numpy as np

# Apply log1p transformation to reduce right skewness.
#
# log1p(x) computes log(1 + x), making it safe for zero values
# while compressing extremely large compensation observations.
df["ConvertedCompYearly_Log"] = np.log1p(
    df["ConvertedCompYearly"]
)

# Compare skewness before and after transformation.
skewness_comparison = pd.DataFrame({
    "Variable": [
        "ConvertedCompYearly",
        "ConvertedCompYearly_Log"
    ],
    "Skewness": [
        df["ConvertedCompYearly"].skew(),
        df["ConvertedCompYearly_Log"].skew()
    ]
})

display(skewness_comparison)

,Variable,Skewness
0,ConvertedCompYearly,87.708258
1,ConvertedCompYearly_Log,-4.281682


### Log Transformation Analysis

The original `ConvertedCompYearly` distribution exhibited extreme positive skewness, with a skewness coefficient of approximately **87.71**, confirming the strong influence of exceptionally high compensation values.

After applying the `log1p()` transformation, skewness decreased substantially but shifted to approximately **-4.28**. This indicates that the logarithmic transformation successfully reduced the extreme right skew but over-corrected the distribution, resulting in considerable left skewness.

Therefore, the log transformation improves the scale and reduces the influence of extreme high values, but it does not produce an approximately symmetric distribution. Additional investigation of the compensation distribution and its outliers would be appropriate before selecting the transformed feature for statistical modeling.

### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [38]:
# Profile YearsCodePro before creating ExperienceLevel.
# This step identifies numeric values, missing observations,
# and any non-numeric categories that require explicit treatment.

years_code_profile = (
    df["YearsCodePro"]
    .value_counts(dropna=False)
    .rename_axis("YearsCodePro")
    .reset_index(name="Count")
)

display(years_code_profile)

,YearsCodePro,Count
0,NaN,13827
1,2,4168
2,3,4093
3,5,3526
4,10,3251
5,4,3215
6,Less than 1 year,2856
7,6,2843
8,1,2639
9,8,2549


In [39]:
# Convert YearsCodePro temporarily to numeric.
# Invalid or non-numeric values are converted to NaN for inspection only.
years_numeric_test = pd.to_numeric(
    df["YearsCodePro"],
    errors="coerce"
)

non_numeric_mask = (
    df["YearsCodePro"].notna()
    & years_numeric_test.isna()
)

display(
    df.loc[non_numeric_mask, "YearsCodePro"]
      .value_counts()
      .rename_axis("Non_Numeric_Value")
      .reset_index(name="Count")
)

,Non_Numeric_Value,Count
0,Less than 1 year,2856
1,More than 50 years,50


In [40]:
# Create a numeric version of YearsCodePro while preserving
# the original survey variable.
df["YearsCodePro_Numeric"] = (
    df["YearsCodePro"]
    .replace({
        "Less than 1 year": 0.5,
        "More than 50 years": 51
    })
    .pipe(pd.to_numeric, errors="coerce")
)

In [41]:
# Categorize professional coding experience into meaningful groups.
#
# Entry:  less than 2 years
# Junior: 2–4 years
# Mid:    5–9 years
# Senior: 10–19 years
# Expert: 20+ years

experience_bins = [-float("inf"), 2, 5, 10, 20, float("inf")]

experience_labels = [
    "Entry",
    "Junior",
    "Mid",
    "Senior",
    "Expert"
]

df["ExperienceLevel"] = pd.cut(
    df["YearsCodePro_Numeric"],
    bins=experience_bins,
    labels=experience_labels,
    right=False
)

In [42]:
# Validate the engineered feature.
experience_summary = (
    df["ExperienceLevel"]
    .value_counts(dropna=False)
    .rename_axis("ExperienceLevel")
    .reset_index(name="Count")
)

experience_summary["Percentage"] = (
    experience_summary["Count"] / len(df) * 100
).round(2)

display(experience_summary)

,ExperienceLevel,Count,Percentage
0,NaN,13827,21.13
1,Senior,13327,20.37
2,Mid,12928,19.76
3,Junior,11476,17.54
4,Expert,8384,12.81
5,Entry,5495,8.40


### Experience Level Feature Engineering Analysis

The `ExperienceLevel` feature was created from `YearsCodePro` to transform raw years of professional coding experience into more interpretable career-level groups.

The resulting distribution shows that **Senior (20.37%)** and **Mid-level (19.76%)** developers represent the largest classified groups, followed by **Junior (17.54%)**, **Expert (12.81%)**, and **Entry-level (8.40%)** respondents. This indicates that the survey contains a substantial proportion of respondents with established professional coding experience.

Approximately **21.13% of records remain unclassified (`NaN`)** because their original `YearsCodePro` value was missing or could not be reliably converted into a numerical value. These observations were intentionally preserved as missing rather than assigning an arbitrary experience category.

Overall, the engineered feature provides a more interpretable representation of professional experience while preserving the uncertainty present in the original data. The resulting categories can now support segmentation and comparative analysis across variables such as employment status, compensation, remote-work arrangements, and other developer characteristics.

### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
